# Unsloth Studio (X8) on a free Colab T4

Clones **https://github.com/GoldodemonINC/unsloth_8X.git**,
installs Unsloth Studio (backend + built frontend), starts it on port 8888 and
shows the UI through **Colab's own port proxy**.

**Runtime -> Change runtime type -> T4 GPU** (the free tier is enough).

1. Check the GPU
2. Clone the repo
3. Install (this builds the frontend - expect a few minutes)
4. Start Studio and open the UI in the cell below
5. Optional: a shareable `https://` link for another device


In [ ]:
import sys, torch

print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("device :", props.name, "- %.1f GB" % (props.total_memory / 1024 ** 3))


## 2. Clone the repo

In [ ]:
REPO_URL  = "https://github.com/GoldodemonINC/unsloth_8X.git"
BRANCH   = "main"
REPO_DIR = "/content/unsloth_8X"

# Private repo? Keep a token in Colab Secrets (GITHUB_TOKEN) and uncomment:
# from google.colab import userdata
# CLONE_URL = REPO_URL.replace("https://", "https://%s@" % userdata.get("GITHUB_TOKEN"))
CLONE_URL = REPO_URL

import shutil
shutil.rmtree(REPO_DIR, ignore_errors=True)
!git clone --depth 1 --branch {BRANCH} {CLONE_URL} {REPO_DIR}
!cd {REPO_DIR} && git log --oneline -1 && ls studio


## 3. Install Unsloth Studio

Backend dependencies go into Colab's system Python (no venv on Colab) and the
frontend is built into `studio/frontend/dist`, which is what the server serves.


In [ ]:
%%bash
cd /content/unsloth_8X
chmod +x studio/setup.sh
# whisper.cpp / audio.cpp are not needed for training or merging and only slow
# the install down on Colab, so they are skipped.
UNSLOTH_SKIP_WHISPER_INSTALL=1 UNSLOTH_SKIP_AUDIO_CPP_INSTALL=1 bash studio/setup.sh --local


## 4. Start Studio and open the UI

The cell below starts the backend on port 8888 and embeds the UI in the cell
output through Colab's port proxy. Log in as **`unsloth`** with the password it
prints.

In [ ]:
import sys

REPO_DIR = "/content/unsloth_8X"
BACKEND  = f"{REPO_DIR}/studio/backend"
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)

import colab

PORT = 8888

# 1) Give the admin account a real password and print it (user: unsloth).
creds = colab._finalize_colab_admin_password()
if creds:
    colab._show_colab_login_credentials(*creds)
else:
    print("Admin password already set - log in as 'unsloth' with the password you chose.")

# 2) Colab's port-proxy URL for this port (informational).
print("Colab proxy:", colab.get_colab_url(PORT))

# 3) Start the server and embed the UI in this cell.
colab.start(port=PORT, cloudflare=False)


## 5. Optional: a shareable link for another device

Colab's proxy URLs are session-scoped and often stay blank in an embedded
frame, so a Cloudflare quick tunnel is the reliable way to open Studio on a
phone or in a normal browser tab.

If the start cell above is still running, interrupt it first
(**Kernel -> Interrupt**); the server keeps serving in its own thread.

In [ ]:
colab.start(port=8888, cloudflare=True)


## 6. Smoke test the Hub path used by Merging Studio

This is the same code the Merging page calls: it reads only `config.json` from
the Hub, so it is quick and downloads no weights.

In [ ]:
import sys
sys.path.insert(0, "/content/unsloth_8X/studio/backend")
from core.advanced import merge

repo = "unsloth/Llama-3.2-1B"
print("repo id :", merge.hf_repo_id(repo))
print("spec    :", merge.probe_model_spec(f"hf://{repo}"))


### Troubleshooting

* **"Frontend not built"** - re-run the install cell (step 3).
* **Blank iframe** - interrupt the start cell and run the Cloudflare cell (step 5).
* **Port already in use** - set `PORT = 8889` in the start cell before calling `start()`.
* **Restarting** - *Runtime -> Restart runtime*, then run the cells again.
* **Testing the merge search** - in Studio open **Advanced -> Merging**, leave each
  slot in **Hugging Face** mode and type e.g. `llama 3.2 1b`. The list comes from
  the Hub, the same search the trainer uses.
